# Memory Tool Agent

In this notebook, I build an agent that can remember a fact in one `run()` call and recall it in a completely separate one, using the Hugging Face `smolagents` library.

By default, calling `agent.run()` again starts the agent's own reasoning fresh, it does not remember what it said or did the last time. That is `reset=True`, and it is the default. So if I want an agent to carry a fact across two separate `run()` calls, the agent's own conversation history is not where that fact can live, it needs somewhere else to go.

In this notebook, I will:

- Write a plain key-value memory store as ordinary Python
- Handle asking for a fact that was never stored
- Turn storing and recalling into two tools
- Give an agent both tools, store a fact in one `run()` call, and recall it in a completely separate one
- Look at why that works, and where a memory store this simple falls apart

## 1. Importing Libraries and Creating the Model

First I import the pieces I need from `smolagents`.

`CodeAgent` is the agent that writes and runs Python code to solve a task. `tool` is the decorator I use to turn a plain function into something an agent can call. `InferenceClientModel` is the language model, which runs on the Hugging Face Inference API rather than on my own machine.

In [ ]:
from smolagents import CodeAgent, InferenceClientModel, tool

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct"
)

## 2. Writing a Basic Key-Value Memory Store

Before I build a tool, I write the storage as an ordinary Python dictionary and two plain functions around it. `MEMORY` is defined once, here, and every function in this notebook that touches it shares the exact same dictionary, which is the detail this whole notebook rests on.

In [ ]:
MEMORY = {}


def remember(key: str, value: str) -> str:
    """Stores a value under a key in memory."""
    MEMORY[key] = value
    return f"Stored '{value}' under '{key}'."

## 3. Writing a Basic Recall Function

A store is only useful if I can get things back out of it, so I write the other half.

In [ ]:
def recall(key: str) -> str:
    """Returns the value stored under a key."""
    return MEMORY[key]

## 4. Testing Remembering and Recalling a Fact

I try storing one fact and reading it straight back, before anything about tools or agents gets involved.

In [ ]:
print(remember("favorite_language", "Python"))
print(recall("favorite_language"))